In [1]:
import itertools
import numpy as np


def level_to_level_connectivity(graph):
    if isinstance(graph, dict):
        conn = []
        for k, v in graph.items():
            conn.append(start_to_end_connectivity(v))
        return conn
    else:
        if isinstance(graph, (list, tuple, set)):
            return list(graph)
        else:
            utils.raise_value_err(
                "The last level of the groups graph has to be a list."
            )


def time_scales(graph, ts={}):
    levels = depth(graph)
    conn = connectivity(graph)
#     if levels == 0:
#         ts = [list(conn)]
#     else:
    for l in range(-1,levels):
        ts[l+1] = merge(conn, it_max=l+1)
    return ts


def connectivity(graph):
    if isinstance(graph, dict):
        conn = []
        for k, v in graph.items():
            conn.append(connectivity(v))
        return conn
    else:
        if isinstance(graph, (list, tuple, set)):
            return list(graph)
        else:
            raise ValueError(
                "The last level of the groups connectivity has to be a list."
            )

            
def merge(ls, it=0, it_max=1):
    if any(isinstance(l, list) for l in ls):
        if it < it_max:
            ls = list(itertools.chain.from_iterable(ls))
            return merge(ls, it=it+1, it_max=it_max)
        else:
            return join(ls)

def join(ls):
    try:
        ls = [list(itertools.chain.from_iterable(l)) for l in ls]
        return join(ls)
    except:
        return ls

        



def depth(d):
    if isinstance(d, dict):
        return 1 + (max(map(depth, d.values())) if d else 0)
    return 0

In [2]:

graph = [0,0,0,0]

graph[0] = [0]

graph[1] = {
    0: [0,1,2]
}

graph[2] = {
    0: {
        0: [0,1,2],
        1: [3,4,5],
        2: [6,7,8]
    }
}

graph[3] = {
    1: {
        1: {
            1: [1,2,3],
            2: [4,5,6],
            3: [7,8,9]
        },
        2: {
            4: [10,11,12],
            5: [13,14,15],
            6: [16,17,18]
        },
        3: {
            7: [19,20,21],
            8: [22,23,24],
            9: [25,26,27]
        }
    }
}

    
def con(graph, level=0, conn={}):
    d = {}
    conn[level] = []
    if isinstance(graph, dict):
        for g in graph.values():
            if hasattr(g, 'keys'):
                l = list(g.keys())
                d = {**d, **g}
            else:
                l = d = g
            conn[level].append(l)
    else:
        conn[level] = None
    if d != {}:
        return con(d, level=level+1, conn=conn)
    else:
        return conn

    
# for i in graph:
#     print(depth(i))
#     c = con(i)
#     print(c)
#     print()
    

In [3]:
for i in graph:
    ts = time_scales(i)
    print(ts)
    print()

{0: None}

{0: [[0, 1, 2]], 1: None}

{0: [[0, 1, 2, 3, 4, 5, 6, 7, 8]], 1: [[0, 1, 2], [3, 4, 5], [6, 7, 8]], 2: None}

{0: [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]], 1: [[1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 16, 17, 18], [19, 20, 21, 22, 23, 24, 25, 26, 27]], 2: [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12], [13, 14, 15], [16, 17, 18], [19, 20, 21], [22, 23, 24], [25, 26, 27]], 3: None}



In [4]:
import tensorflow as tf
import tensorflow.keras.layers as tf_layers


In [5]:
c = con(graph[2])

feat = [2,3,9]

y = [ tf.random.normal([10, i]) for i in feat ]
    

In [6]:
print(y)
y[0][0,1]*y[1][0,0]*y[2][0,0]

print(tf.split(y[0], num_or_size_splits=[2], axis=1))

[<tf.Tensor: shape=(10, 2), dtype=float32, numpy=
array([[ 0.24729043,  1.1381189 ],
       [-1.1489819 , -2.5243342 ],
       [-1.1015497 ,  1.7268312 ],
       [ 1.4253043 ,  1.8475227 ],
       [ 0.3289835 ,  1.8045729 ],
       [-0.06394394, -0.18001445],
       [-0.09107347,  0.49042988],
       [ 1.1921587 , -0.2902357 ],
       [-0.01917436, -0.41200545],
       [-0.9653217 , -2.3898633 ]], dtype=float32)>, <tf.Tensor: shape=(10, 3), dtype=float32, numpy=
array([[ 1.6926231 ,  0.4103731 ,  0.682581  ],
       [ 1.0302086 ,  1.2042323 , -0.6177676 ],
       [ 0.51064444, -2.2322938 , -1.4171661 ],
       [ 0.87004423, -1.2034478 , -0.36371362],
       [ 0.31493074, -1.0591313 ,  0.42183664],
       [ 0.07810088, -2.8421674 , -0.08407307],
       [ 0.5508971 , -0.6947362 , -0.10430209],
       [-1.1281558 ,  0.4892998 ,  0.37555236],
       [ 1.9406401 , -1.4849521 , -2.0185058 ],
       [ 0.05977228, -0.18212686, -0.99044067]], dtype=float32)>, <tf.Tensor: shape=(10, 9), dtype=fl

In [7]:
for l, indeces in c.items():
    if indeces:
        y_slow = tf.split(y[l], num_or_size_splits=y[l].shape[1], axis=1)
        y_fast = [ tf.gather(y[l+1], idx, axis=1) for idx in indeces ]
        if len(y_slow) != len(y_fast):
            y_slow = y_slow[-len(y_fast):]
        y_mul = [ tf_layers.Multiply()([ys, yf]) for ys, yf in zip(y_slow, y_fast) ]
        y[l+1] = tf_layers.Concatenate()(y_mul) if len(y_mul) > 1 else y_mul[-1]
print(y)

[<tf.Tensor: shape=(10, 2), dtype=float32, numpy=
array([[ 0.24729043,  1.1381189 ],
       [-1.1489819 , -2.5243342 ],
       [-1.1015497 ,  1.7268312 ],
       [ 1.4253043 ,  1.8475227 ],
       [ 0.3289835 ,  1.8045729 ],
       [-0.06394394, -0.18001445],
       [-0.09107347,  0.49042988],
       [ 1.1921587 , -0.2902357 ],
       [-0.01917436, -0.41200545],
       [-0.9653217 , -2.3898633 ]], dtype=float32)>, <tf.Tensor: shape=(10, 3), dtype=float32, numpy=
array([[ 1.9264063 ,  0.46705335,  0.7768583 ],
       [-2.6005907 , -3.0398848 ,  1.5594518 ],
       [ 0.8817967 , -3.8547947 , -2.4472067 ],
       [ 1.6074265 , -2.2233973 , -0.6719692 ],
       [ 0.5683155 , -1.9112797 ,  0.761235  ],
       [-0.01405929,  0.5116312 ,  0.01513437],
       [ 0.2701764 , -0.34071937, -0.05115286],
       [ 0.32743108, -0.14201227, -0.1089987 ],
       [-0.7995543 ,  0.61180836,  0.8316354 ],
       [-0.14284758,  0.4352583 ,  2.3670177 ]], dtype=float32)>, <tf.Tensor: shape=(10, 9), dtype=fl

In [8]:
T_norm = tf.random.normal([10, 1])
rho = tf.random.normal([10, 1])
Y = tf.random.normal([10, 1])
print(Y)

g_idx = time_scales(graph[0])
print(g_idx)

Y_grouped = []
if g_idx[0]:
    for indeces in g_idx[0]:
        Y_grouped.append(
            tf.math.reduce_sum(
                tf.gather(Y, indeces, axis=1), axis=1, keepdims=True
            )
        )
    ic = tf_layers.Concatenate()([T_norm, rho, *Y_grouped])
    print(ic)

tf.Tensor(
[[ 0.40008888]
 [ 0.55357236]
 [ 0.545298  ]
 [-1.5599333 ]
 [ 0.464718  ]
 [-0.39469752]
 [-0.45255527]
 [-0.760178  ]
 [ 0.44859388]
 [-0.6083784 ]], shape=(10, 1), dtype=float32)
{0: None, 1: [[1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 16, 17, 18], [19, 20, 21, 22, 23, 24, 25, 26, 27]], 2: [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12], [13, 14, 15], [16, 17, 18], [19, 20, 21], [22, 23, 24], [25, 26, 27]], 3: None}


In [9]:
tf.concat(y, axis=-1)

<tf.Tensor: shape=(10, 14), dtype=float32, numpy=
array([[ 2.4729043e-01,  1.1381189e+00,  1.9264063e+00,  4.6705335e-01,
         7.7685833e-01, -1.7839587e+00, -1.1526740e+00,  2.7861819e+00,
         4.1327253e-01,  3.3745787e-01, -1.5558225e-01,  3.2766923e-01,
        -1.4578788e-01, -2.9485604e-01],
       [-1.1489819e+00, -2.5243342e+00, -2.6005907e+00, -3.0398848e+00,
         1.5594518e+00, -2.2435541e+00,  8.5165710e+00, -1.5784358e+00,
        -1.8645651e+00,  3.9763529e+00,  6.0393828e-01, -1.7147871e+00,
        -8.4783876e-01, -6.9895786e-01],
       [-1.1015497e+00,  1.7268312e+00,  8.8179672e-01, -3.8547947e+00,
        -2.4472067e+00,  4.5590210e-01,  1.2488261e-01,  7.7200556e-01,
         2.9974976e+00,  7.5406832e-01, -1.1480367e+00, -1.3795199e+00,
         5.5230565e+00, -9.4938087e-01],
       [ 1.4253043e+00,  1.8475227e+00,  1.6074265e+00, -2.2233973e+00,
        -6.7196918e-01,  2.5337011e-01, -8.3053386e-01,  1.2067552e+00,
         9.2423618e-02,  4.1622372e

In [12]:
y = [ tf.random.normal([10, i]) for i in feat ]

for i in y:
    print(i.shape)

cc = con(graph[2])
print(cc)

res = []
for ts, indeces in c.items():
    if indeces:
        dY_fast = tf.concat(
            [
                tf.math.reduce_sum(
                    tf.gather(y[ts+1], idx, axis=1), axis=1, keepdims=True
                ) for idx in indeces
            ], axis=1
        )
        dY_slow = y[ts]
        if dY_slow.shape[1] != dY_fast.shape[1]:
            dY_slow = tf.split(dY_slow, dY_slow.shape[1], axis=1)[-dY_fast.shape[1]:]
        res.append(dY_fast - dY_slow)
        
print(res)

(10, 2)
(10, 3)
(10, 9)
{0: [[0, 1, 2]], 1: [[0, 1, 2], [3, 4, 5], [6, 7, 8]], 2: None}
[<tf.Tensor: shape=(1, 10, 1), dtype=float32, numpy=
array([[[ 3.0700521 ],
        [-0.10264444],
        [ 0.24498421],
        [-0.96353614],
        [-4.872583  ],
        [ 0.0682835 ],
        [ 1.8237647 ],
        [ 4.0238366 ],
        [ 0.7148773 ],
        [ 0.8384251 ]]], dtype=float32)>, <tf.Tensor: shape=(10, 3), dtype=float32, numpy=
array([[ 3.1971133 ,  1.1177855 , -2.4332175 ],
       [ 0.89310014,  4.4586854 ,  3.2120333 ],
       [ 1.0799723 , -0.7666039 ,  1.9636109 ],
       [ 0.1244742 ,  1.6310668 ,  0.3703419 ],
       [ 1.2086008 ,  1.6501517 ,  1.297748  ],
       [-0.2511409 ,  0.0856806 ,  0.34528017],
       [-2.080089  ,  0.49835446, -0.669237  ],
       [-0.06560537, -9.265327  , -1.7692286 ],
       [ 1.9139798 , -0.9267785 , -1.1411713 ],
       [-0.03618272, -1.2288191 ,  0.11257809]], dtype=float32)>]
